# Temperature-dependent 4-wire resistivity

This notebook performs temperature-dependent four-wire resistivity measurements on the Janis probe station. It is set up to use the Scientific Intruments Model 9700 temperature controller and HP 3458A 8.5 digit multimeter. The program sets a desired temperature on the controller, waits for it to stabilize, then collects a preset number of measurements of the resistivity from the DMM with a preset time interval. It then moves on to the next temperature. The raw temperature-resistivity data are saved to a text file.  

## Import libraries
Imports the libraries required to run the program.

In [1]:
import pyvisa
import time
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

## Functions
This section contains function used in the rest of the program.

In [2]:
def connect_to_resources(GPIB_addr):
    """Connects to the multimeter and temperature controller using GPIB with the pyVISA library. 
        The input is a dictionary of instrument names and GPIB addresses as key,value pairs. 
        The output is a dictionary of instrument names and pyVISA resource objects as key,value pairs."""
    resman = pyvisa.ResourceManager()
    resources = {}
    for instr, addr in GPIB_addr.items():
        resources[instr] = resman.open_resource('GPIB0::' + str(addr) + '::INSTR')
        try:
            resources[instr].write('*IDN?')
            instr_ID = resources[instr].query('END 2;ID?')[:-2] #Remove carriage return and line feed characters from the end
            print('Connected at GPIB address ' + str(addr) + ' to instrument with ID: ' + instr_ID)
        except pyvisa.errors.VisaIOError:
            print('Error connecting to instrument at GPIB address ' + str(addr) + '. Please check the GPIB address and end-of-string character.\nProgram execution aborted.')
            raise pyvisa.errors.VisaIOError(-1073807265)
            
    print('All instruments connected.\n')
    return(resources)

def configure_multimeter(instrument, perform_autocal):
    """Configures the digital multimeter for 4-wire ohms measurements. 
        The input is a pyVISA resource object and a Boolean perform_autocal parameter which determines whether
        autocalibration is performed before further configuration. Autocal takes about 10 minutes.
        No output is generated.""" 
    instrument.clear()
    instrument.write('RESET; END 2')
    error_state = instrument.query('ERR?')
    
    if(error_state != '0\r\n'): #If errors are reported
        print('An error was reported by the instrument: ' + instrument.query('ERRSTR?'))
        raise pyvisa.errors.VisaIOError(0)
        
    instrument.write('OHMF,AUTO;OCOMP ON;APER 0.5;LOCK ON;DELAY 0')
    print('4-wire ohms measurement.\nOffset compensation on.\nIntegration window: 0.5 s.\nKeyboard lock on.') 
    
    print('Initializing...')
    time.sleep(10) #This can probably be done more elegantly using events, but it works for now.
    
    if(perform_autocal):
        instrument.write('INBUF ON')
        instrument.write('ACAL DCV')
        print('AUTOCAL running')
        for _ in range(22):
            print('.', end='')
            time.sleep(30)
        instrument.write('INBUF OFF')
        print('Autocalibration complete.')
    
    instrument.timeout = 10000 #To prevent timeout errors
    
    error_state = instrument.query('ERR?')
    if(error_state != '0\r\n'):
        print('An error was reported by the instrument: ' + instrument.query('ERRSTR?'))
        raise pyvisa.errors.VisaIOError(0)
    

def configure_tempcontr(instrument):
    """Configures the temperature controller.
    This function takes a pyVISA resource object as input and returns no output."""
    curr_temp = instrument.query('TA?')[3:-2]
    instrument.write('SET ' + curr_temp) 
    instrument.write('MHP 75.00')
    instrument.write('MODE 2')
    instrument.write('CTYP 1')
    print('\nCurrent temperature: ' + curr_temp + 'K.\nPID control on.\nMax. heater power 75%.')
    
def is_temp_stable(instrument, del_T, num_polls, poll_time_interval, setpoint):
    """This function determines whether the current temperature has stabilized at the setpoint. 
    It takes a pyVISA resource object, the required temperature accuracy, number of polls per setpoint, poll time interval
    and the current setpoint as imputs.
    The function returns True if the temperature is stable, False otherwise."""
    temps = [setpoint]
    for _ in range(3):
        temps.append(float(instrument.query('TA?')[3:-2]))
        time.sleep(poll_time_interval*num_polls*3/3)
    return(max(temps)-min(temps) < del_T)

def measure_temp_step(instrument_temp, instrument_mult, del_T, num_polls, poll_time_interval, setpoint):
    print('Temperature setpoint: ' + str(setpoint))
    instrument_temp.write('SET ' + str(setpoint))
    print('Waiting for temperature stabilization...')
    while not is_temp_stable(instrument_temp, del_T, num_polls, poll_time_interval, setpoint): 
        #Wait until the temperature has stabilized
        time.sleep(30)
    print('Temperature stable.\n')
    
    temp_list = []
    res_list = []
    with open(filename, 'a') as f:
        for _ in range(num_polls):
            temp = float(instrument_temp.query('TA?')[3:-2])
            resist = float(instrument_mult.read()[:-2])
            f.write(str(temp) + ',' + str(resist) + '\n')
            temp_list.append(temp)
            res_list.append(resist)
            time.sleep(poll_time_interval)
    return(temp_list,res_list)

def update_graph(fig, df):
    if(fig.data == ()):
        fig.add_trace(go.Scatter(x=df['Temp'], y=df['Resist'], mode='markers+lines', error_x=
                         dict(type='data',
                              array=df['Tempstd'],
                              visible=True),
                              error_y=
                         dict(type='data',
                              array=df['Resiststd'],
                              visible=True)
                                )
                     )
    else:
        with fig.batch_update():
            fig.data[0]['y'] = df['Resist']
            fig.data[0]['x'] = df['Temp']
            fig.data[0]['error_y'] = dict(type='data',
                                          array=df['Resiststd'],
                                          visible=True)
            fig.data[0]['error_x'] = dict(type='data',
                                          array=df['Tempstd'],
                                          visible=True)
    

## Measurement parameters
This section defines the instrument and program parameters used during the measurements. 

In [4]:
output_folder = 'C:\\Users\\F110216\\Documents\\DATA\\Foelke\\'
if not os.path.exists(output_folder):
    os.mkdir(output_folder)
output_file_name = 'Pw017_4point_PPMS'

#The temperature setpoints must be supplied as an iterator of setpoint values.
temp_setpts =  np.round(list(np.linspace(350,150,101))
                        +
                        list(np.linspace(150,350,101)),3)

#Set the time interval between measurements from the multimeter in seconds. 
#The interval should always be larger than the integration window (0.5 s)
#to prevent double measurements. 2 s is a good default value.
poll_time_interval = 2

#Number of measurements collected from the multimeter per temperature step.
num_polls = 3

#Required temperature accuracy. This value represents the maximum temperature 
#deviation (in K) allowed between three measurements in a time period equal 
#to poll_time_interval*num_polls.
del_T = 0.5

#Set true to perform autocalibration before starting the measurement. 
#Autocal should be run once every 24 hours or after a temperature change of
#more than 1 K. The procedure takes abour 10 minutes.
perform_autocal = False

#GPIB adresses of the temperature controller and multimeter.
GPIB_addr = {'temp':15, 'mult':22}

## Instrument setup
This section sets up the instruments for the measurements.

In [5]:
try:
    resources = {}
    resman = pyvisa.ResourceManager()
    resources['tempresman.open_resource('GPIB0::15::INSTR')
    resman.open_resource('GPIB1::17::INSTR')
    configure_multimeter(resources['mult'], perform_autocal)
    configure_tempcontr(resources['temp'])
except Exception as e:
    print('\nInstrument setup failed.')
else:
    print('\nInstrument setup successful.')

Error connecting to instrument at GPIB address 15. Please check the GPIB address and end-of-string character.
Program execution aborted.

Instrument setup failed.


## Plot
Create the real-time plot of the resistance. 

In [63]:
df = pd.DataFrame(columns=['Temp', 'Tempstd', 'Resist', 'Resiststd'], index=temp_setpts) #For plotting
fig = go.FigureWidget()
fig.update_xaxes(title='Temperature (K)')
fig.update_yaxes(title='Resistance (Ohm)')
fig

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'xaxis': {'title': {'text': 'T…

## Measurement loop
This section contains the main measurement loop. 

In [64]:
filename_counter = 0
filename = output_folder + output_file_name + '_' + str(filename_counter) + '.txt'
while os.path.isfile(filename):
    filename_counter += 1
    filename = output_folder + output_file_name + '_' + str(filename_counter) + '.txt'
else:
    print('Filename: ' + filename, end='\n')
    resources['mult'].write('LOCK ON')
    with open(filename, 'w') as f:
        f.write('Temperature(K),Resistance(Ohm)\n')
    
    
    for setpoint in temp_setpts:
        temp_list,res_list = measure_temp_step(resources['temp'], resources['mult'], del_T, num_polls, poll_time_interval, setpoint)
        df['Temp'][setpoint],df['Resist'][setpoint],df['Tempstd'][setpoint],df['Resiststd'][setpoint]=\
        np.mean(temp_list),np.mean(res_list),np.std(temp_list),np.std(res_list)
        update_graph(fig, df)
    
    
    print('\nMeasurement finished.')  
resources['mult'].write('LOCK OFF');

Filename: C:\Users\F110216\Documents\DATA\EV112_not_reduced\Heating_Cooling_10-300K_5Kperstep reversed_0.txt
Temperature setpoint: 10.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 15.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 20.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 25.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 30.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 35.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 40.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 45.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 50.0
Waiting for temperature stabilization...
Temperature stable.

Temperature setpoint: 55.0
Waiting for temperature stabilization...
Temperature stable.

T

KeyboardInterrupt: 